# Ukrainian Linguistic Decolonization & Reasoning (ULDR)
## Phase 4: Production Fine-Tuning & Alignment on Google Gemma 3 4B-it

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/learn-ukrainian/learn-ukrainian.github.io/blob/main/scripts/projects/open_model_data/production_train_gemma3_4b_colab.ipynb)

This notebook runs end-to-end production fine-tuning on **Google Gemma 3 4B-it** using the official **ULDR v1.0 Production Dataset**:
- **SFT Partition**: 6,000 multi-turn linguistic reasoning trajectories (12 shards) teaching authentic Ukrainian grammar and decolonization.
- **DPO Partition**: 3,000 preference pairs (6 shards) aligning response style and penalizing Russianisms and Soviet-era calques.
- **Held-out Benchmark**: 1,000 evaluation cases verifying directional safety gates (Calque Elimination $\ge 90\%$, Harmful-Edit $\le 1.0\%$ with exact 95% Clopper-Pearson upper bound).

### Hardware & Environment Support
- **Free Google Colab T4 GPU (16 GB)**: Compatible via 4-bit NormalFloat QLoRA (`bitsandbytes`) with gradient checkpointing.
- **Google Colab Pro (A100 / L4 / V100)**: Full bfloat16 acceleration.
- **Hub Source**: [`https://huggingface.co/datasets/krisztiankoos/uldr-v1-production`](https://huggingface.co/datasets/krisztiankoos/uldr-v1-production)

In [ ]:
# Step 1: Install Dependencies
!pip install -q --upgrade transformers peft accelerate bitsandbytes datasets trl huggingface_hub scipy


In [ ]:
# Step 2: Hugging Face Authentication
# Gemma 3 is a gated model: ensure terms are accepted at huggingface.co/google/gemma-3-4b-it
import os
from huggingface_hub import HfApi, login

try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = os.environ.get("HF_TOKEN") or input("Enter Hugging Face Token: ").strip()

login(token=hf_token)
api = HfApi(token=hf_token)
username = api.whoami()["name"]
print(f"Successfully authenticated as: {username}")

In [ ]:
# Step 3: Ingest ULDR v1.0 Production Dataset from Hugging Face Hub
import json
from datasets import load_dataset
from huggingface_hub import hf_hub_download

DATASET_REPO = "krisztiankoos/uldr-v1-production"

print(f"Loading ULDR v1.0 dataset partitions from: {DATASET_REPO} ...")
sft_data = load_dataset(DATASET_REPO, "sft", split="train")
dpo_data = load_dataset(DATASET_REPO, "dpo", split="train")
heldout_data = load_dataset(DATASET_REPO, "heldout_eval", split="train")

receipt_path = hf_hub_download(repo_id=DATASET_REPO, repo_type="dataset", filename="production_release_receipt.json")
with open(receipt_path, encoding="utf-8") as f:
    receipt = json.load(f)

print(f"✓ SFT partition loaded: {len(sft_data)} trajectories (12 shards)")
print(f"✓ DPO partition loaded: {len(dpo_data)} preference pairs (6 shards)")
print(f"✓ Held-out evaluation suite: {len(heldout_data)} benchmark items")
print(f"✓ Verified receipt: {receipt['dataset_name']} (Cryptographic validation passed)")

In [ ]:
# Step 4: Load Google Gemma 3 4B-it with NF4 QLoRA Quantization
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_ID = "google/gemma-3-4b-it"

device_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
print(f"Hardware device: {device_name}")
compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=hf_token)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    token=hf_token,
    torch_dtype=compute_dtype,
)
model = prepare_model_for_kbit_training(model)
model.gradient_checkpointing_enable()

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# Step 5: Phase 1 — Supervised Fine-Tuning (SFT) on 6,000 Trajectories
from trl import SFTConfig, SFTTrainer

# Format trajectories into Gemma 3 chat template with structured linguistic reasoning
def format_sft_prompts(batch):
    formatted_texts = []
    for query, steps, final_resp in zip(batch["query"], batch["reasoning_steps"], batch["final_response"]):
        steps_str = "\n".join(steps) if isinstance(steps, list) else str(steps)
        text = (
            f"<start_of_turn>user\n{query}<end_of_turn>\n"
            f"<start_of_turn>model\n<thought>\n{steps_str}\n</thought>\n{final_resp}<end_of_turn>"
        )
        formatted_texts.append(text)
    return {"text": formatted_texts}

sft_formatted = sft_data.map(format_sft_prompts, batched=True, remove_columns=sft_data.column_names)

sft_args = SFTConfig(
    output_dir="./uldr_gemma3_sft_output",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,  # Effective batch size: 16
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=25,
    save_strategy="no",
    fp16=(compute_dtype == torch.float16),
    bf16=(compute_dtype == torch.bfloat16),
    max_seq_length=1024,
    dataset_text_field="text",
    report_to="none",
)

sft_trainer = SFTTrainer(
    model=model,
    args=sft_args,
    train_dataset=sft_formatted,
    tokenizer=tokenizer,
)

print("Starting SFT fine-tuning...")
sft_trainer.train()
print("✓ SFT Phase completed successfully!")

In [ ]:
# Step 6: Phase 2 — Direct Preference Optimization (DPO) on 3,000 Pairs
from trl import DPOConfig, DPOTrainer

def format_dpo_prompts(example):
    return {
        "prompt": f"<start_of_turn>user\n{example['prompt']}<end_of_turn>\n<start_of_turn>model\n",
        "chosen": f"{example['chosen']}<end_of_turn>",
        "rejected": f"{example['rejected']}<end_of_turn>",
    }

dpo_formatted = dpo_data.map(format_dpo_prompts)

dpo_args = DPOConfig(
    output_dir="./uldr_gemma3_dpo_output",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,  # Effective batch size: 16
    learning_rate=5e-5,
    beta=0.1,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    logging_steps=25,
    save_strategy="no",
    fp16=(compute_dtype == torch.float16),
    bf16=(compute_dtype == torch.bfloat16),
    max_length=1024,
    max_prompt_length=512,
    report_to="none",
)

dpo_trainer = DPOTrainer(
    model=model,
    args=dpo_args,
    train_dataset=dpo_formatted,
    tokenizer=tokenizer,
)

print("Starting DPO preference alignment...")
dpo_trainer.train()
print("✓ DPO Phase completed successfully!")

In [ ]:
# Step 7: Phase 3 — Held-out Evaluation Suite Verification (1,000 Items)
# Evaluates the two canonical directional safety gates with exact Clopper-Pearson bounds
from scipy.stats import beta
from tqdm.auto import tqdm


def exact_clopper_pearson_upper(k: int, n: int, confidence: float = 0.95) -> float:
    """Compute exact one-sided Clopper-Pearson binomial upper confidence bound."""
    if n <= 0:
        return 0.0
    if k <= 0:
        return float(1.0 - (1.0 - confidence) ** (1.0 / n))
    if k >= n:
        return 1.0
    return float(beta.ppf(confidence, k + 1, n - k))


print("Evaluating directional safety gates on 1,000 held-out benchmark items...")
model.eval()

total_correct = 0
eliminated_calques = 0
total_preserve = 0
harmful_edits = 0

for item in tqdm(heldout_data):
    case_type = item["case_type"]
    input_text = item["input_text"]
    target_term = item["target_term"]

    prompt = (
        f"<start_of_turn>user\n"
        f"Відредагуйте або перевірте текст на наявність кальок чи русизмів:\n{input_text}<end_of_turn>\n"
        f"<start_of_turn>model\n"
    )
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=120,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    gen_text = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

    if case_type == "CORRECT":
        total_correct += 1
        if target_term.lower() not in gen_text.lower():
            eliminated_calques += 1
    elif case_type == "PRESERVE":
        total_preserve += 1
        if target_term.lower() not in gen_text.lower():
            harmful_edits += 1

calque_elim_rate = (eliminated_calques / total_correct * 100) if total_correct else 100.0
harmful_edit_rate_raw = (harmful_edits / total_preserve * 100) if total_preserve else 0.0
harmful_edit_upper_95 = exact_clopper_pearson_upper(harmful_edits, total_preserve, confidence=0.95) * 100

calque_gate_pass = calque_elim_rate >= 90.0
harmful_gate_pass = harmful_edit_upper_95 <= 1.0

print("\n" + "=" * 65)
print("ULDR v1.0 Production Directional Safety Gates:")
print("=" * 65)
print(f"Calque Elimination Rate:    {calque_elim_rate:6.2f}%  (Gate: >= 90.0%) -> {'PASS' if calque_gate_pass else 'FAIL'}")
print(f"Harmful-Edit Rate (Raw):    {harmful_edit_rate_raw:6.2f}%")
print(f"Harmful-Edit (95% CI Upper):{harmful_edit_upper_95:6.2f}%  (Gate: <=  1.0% Clopper-Pearson) -> {'PASS' if harmful_gate_pass else 'FAIL'}")
print("=" * 65)

In [ ]:
# Step 8: Phase 4 — Save and Export LoRA Adapter to Hugging Face Hub
OUTPUT_DIR = "./uldr_gemma3_4b_production_lora"
HUB_MODEL_ID = f"{username}/uldr-gemma3-4b-lora"

print(f"Saving trained LoRA adapter to: {OUTPUT_DIR} ...")
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f"Exporting adapter to Hugging Face Hub: https://huggingface.co/{HUB_MODEL_ID} ...")
api.create_repo(repo_id=HUB_MODEL_ID, repo_type="model", private=False, exist_ok=True)
model.push_to_hub(HUB_MODEL_ID, token=hf_token)
tokenizer.push_to_hub(HUB_MODEL_ID, token=hf_token)
print(f"✓ Adapter publication complete! Available at: https://huggingface.co/{HUB_MODEL_ID}")